# Preprocessing without interpolation

Run this notebook in VS Code from this repository or its scripts folder. It reads the five annual ZIP files from data/raw and writes data/processed/albemarle_pm25_corrected_poc_without_interpolation.csv. Missing measurements remain blank.

Select a Python kernel with pandas and NumPy installed. Restart the kernel and run all cells in order to verify a clean run.

Measurement selection updated: use EPA method 736 (POC 23) before 2023-08-01 and method 636 (POC 3) afterward. Original POC-3 output is preserved for comparison. This is retrospective forecasting with revised historical measurements, not an exact reconstruction of data availability in 2021-2023.

Source: https://www.epa.gov/system/files/documents/2024-05/quick-reference_t640-data_final_05-13-2024.pdf


In [1]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path


List the files uploaded

In [2]:
# Locate the repository from the notebook folder or the repository root.
working_dir = Path.cwd().resolve()
project_root = next(
    (folder for folder in (working_dir, *working_dir.parents)
     if (folder / "data" / "raw").is_dir()
     and (folder / "scripts").is_dir()),
    None
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the DS4002-project2 repository.")

raw_dir = project_root / "data" / "raw"
files = [raw_dir / f"daily_88101_{year}.zip" for year in range(2021, 2026)]
missing_files = [str(filename) for filename in files if not filename.is_file()]
if missing_files:
    raise FileNotFoundError("Missing input ZIP files: " + ", ".join(missing_files))

yearly_data = []
print("Input folder:", raw_dir)


Input folder: C:\Users\annie\OneDrive\Desktop\DS4002-project2\data\raw


Read the csv inside each zip file

In [3]:
for filename in files:
    with zipfile.ZipFile(filename, "r") as z:
        csv_files = [
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ]

        if not csv_files:
            raise ValueError(f"No CSV file found inside {filename}")

        daily = pd.read_csv(z.open(csv_files[0]), low_memory=False)
        yearly_data.append(daily)

        print(filename.name, "loaded:", daily.shape)

daily_88101_2021.zip loaded: (790468, 29)
daily_88101_2022.zip loaded: (863466, 29)
daily_88101_2023.zip loaded: (847162, 29)
daily_88101_2024.zip loaded: (740924, 29)
daily_88101_2025.zip loaded: (758834, 29)


Combine all five years

In [4]:
epa_data = pd.concat(yearly_data, ignore_index=True)

print("\nCombined dataset shape:", epa_data.shape)
print("\nColumns:")
print(epa_data.columns.tolist())


Combined dataset shape: (4000854, 29)

Columns:
['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Sample Duration', 'Pollutant Standard', 'Date Local', 'Units of Measure', 'Event Type', 'Observation Count', 'Observation Percent', 'Arithmetic Mean', '1st Max Value', '1st Max Hour', 'AQI', 'Method Code', 'Method Name', 'Local Site Name', 'Address', 'State Name', 'County Name', 'City Name', 'CBSA Name', 'Date of Last Change']


Select one consistent EPA-adjusted measurement series: POC 23/method 736 before August 1, 2023; POC 3/method 636 from that date onward. This replaces the original POC-3-only rule.


In [5]:
# Select the site and daily summaries of hourly PM2.5 measurements.
site_mask = (
    (epa_data["State Code"] == 51) &
    (epa_data["County Code"] == 3) &
    (epa_data["Site Num"] == 1) &
    (epa_data["Parameter Code"] == 88101) &
    (epa_data["Sample Duration"] == "1 HOUR")
)
site_data = epa_data.loc[site_mask].copy()
record_dates = pd.to_datetime(site_data["Date Local"], errors="coerce")
cutoff = pd.Timestamp("2023-08-01")

# EPA method 736 is the retrospective correction of older method 236 data.
# Method 636 already applies Network Data Alignment at collection time.
# Choose existing EPA values; do not calculate corrections or interpolate gaps.
older_corrected = (
    (record_dates < cutoff) &
    (site_data["POC"] == 23) &
    (site_data["Method Code"] == 736)
)
newer_aligned = (
    (record_dates >= cutoff) &
    (site_data["POC"] == 3) &
    (site_data["Method Code"] == 636)
)
data = site_data.loc[older_corrected | newer_aligned].copy()
selected_methods = data[["POC", "Method Code", "Method Name"]].drop_duplicates()

print("Rows after filtering:", len(data))
print("Unique dates:", data["Date Local"].nunique())
print("Selected measurement versions:")
print(selected_methods.to_string(index=False))
print("Event labels in selected records:")
print(data.groupby(["Method Code", "Event Type"], dropna=False).size())


Rows after filtering: 1780
Unique dates: 1779
Selected measurement versions:
 POC  Method Code                                                                        Method Name
  23          736                      Teledyne T640 at 5.0 LPM (Corrected) - Broadband spectroscopy
   3          636 Teledyne T640 at 5.0 LPM w/Network Data Alignment enabled - Broadband spectroscopy
Event labels in selected records:
Method Code  Event Type
636          Excluded        1
             Included        1
             NaN           874
736          Included        1
             NaN           903
dtype: int64


Clean the relevant columns (Convert dates and numeric columns to the correct types)

In [6]:
data["Date Local"] = pd.to_datetime(
    data["Date Local"], errors="coerce"
)

data["Arithmetic Mean"] = pd.to_numeric(
    data["Arithmetic Mean"], errors="coerce"
)

data["Observation Count"] = pd.to_numeric(
    data["Observation Count"], errors="coerce"
)

data["Observation Percent"] = pd.to_numeric(
    data["Observation Percent"], errors="coerce"
)

Remove records that do not have a usable date or PM2.5 mean

In [7]:
data = data.dropna(
    subset=["Date Local", "Arithmetic Mean"]
).copy()

print("Rows after cleaning dates and means:", len(data))

Rows after cleaning dates and means: 1780


Keep dates with at least 18 hourly observations

In [8]:
data = data[
    data["Observation Count"] >= 18
].copy()

Assign priority to the event types

In [9]:
data["Event Priority"] = data["Event Type"].map({
    "Included": 2,
    "Excluded": 0
}).fillna(1)

Put the preferred record first for each date

In [10]:
data = data.sort_values(
    ["Date Local", "Event Priority"],
    ascending=[True, False]
)

Keep one record per date

In [11]:
data = data.drop_duplicates(
    subset=["Date Local"],
    keep="first"
)

data = data.sort_values("Date Local").copy()

print("Usable measurement dates:", len(data))
print(
    "Duplicate dates remaining:",
    data["Date Local"].duplicated().sum()
)

print("\nEvent types selected:")
print(data["Event Type"].value_counts(dropna=False))

Usable measurement dates: 1759
Duplicate dates remaining: 0

Event types selected:
Event Type
NaN         1757
Included       2
Name: count, dtype: int64


Keep the columns needed for the training stage (Needs the date and PM2.5 measurement. The additional columns help the team document measurement quality and method changes)

In [12]:
data = data[[
    "Date Local",
    "Arithmetic Mean",
    "Observation Count",
    "Observation Percent",
    "Event Type",
    "Method Code",
    "Method Name"
]].copy()

 Rename columns to make them easier to use in Python

In [13]:
data = data.rename(columns={
    "Date Local": "Date",
    "Arithmetic Mean": "PM2.5",
    "Observation Count": "Observation_Count",
    "Observation Percent": "Observation_Percent",
    "Event Type": "Event_Type",
    "Method Code": "Method_Code",
    "Method Name": "Method_Name"
})

print(data.head())

             Date     PM2.5  Observation_Count  Observation_Percent  \
720863 2021-01-07  5.054167                 24                100.0   
720864 2021-01-08  6.900000                 24                100.0   
720865 2021-01-09  4.562500                 24                100.0   
720866 2021-01-10  4.216667                 24                100.0   
720867 2021-01-11  8.600000                 24                100.0   

       Event_Type  Method_Code  \
720863        NaN          736   
720864        NaN          736   
720865        NaN          736   
720866        NaN          736   
720867        NaN          736   

                                              Method_Name  
720863  Teledyne T640 at 5.0 LPM (Corrected) - Broadba...  
720864  Teledyne T640 at 5.0 LPM (Corrected) - Broadba...  
720865  Teledyne T640 at 5.0 LPM (Corrected) - Broadba...  
720866  Teledyne T640 at 5.0 LPM (Corrected) - Broadba...  
720867  Teledyne T640 at 5.0 LPM (Corrected) - Broadba...  


Create every date in the project period

In [14]:
calendar = pd.date_range(
    start="2021-01-01",
    end="2025-12-31",
    freq="D"
)

Align the measurements to the complete calendar

In [15]:
data = (
    data.set_index("Date")
        .reindex(calendar)
        .rename_axis("Date")
        .reset_index()
)

print("Total calendar dates:", len(data))
print("Missing PM2.5 values:", data["PM2.5"].isna().sum())
print("\nFirst five rows:")
print(data.head())

print("\nLast five rows:")
print(data.tail())

Total calendar dates: 1826
Missing PM2.5 values: 67

First five rows:
        Date  PM2.5  Observation_Count  Observation_Percent Event_Type  \
0 2021-01-01    NaN                NaN                  NaN        NaN   
1 2021-01-02    NaN                NaN                  NaN        NaN   
2 2021-01-03    NaN                NaN                  NaN        NaN   
3 2021-01-04    NaN                NaN                  NaN        NaN   
4 2021-01-05    NaN                NaN                  NaN        NaN   

   Method_Code Method_Name  
0          NaN         NaN  
1          NaN         NaN  
2          NaN         NaN  
3          NaN         NaN  
4          NaN         NaN  

Last five rows:
           Date     PM2.5  Observation_Count  Observation_Percent Event_Type  \
1821 2025-12-27  6.525000               24.0                100.0        NaN   
1822 2025-12-28  7.441667               24.0                100.0        NaN   
1823 2025-12-29  3.979167               24.0          

Count missing PM2.5 values by year

In [16]:
data["Year"] = data["Date"].dt.year

print("Missing PM2.5 dates by year:")
print(
    data.groupby("Year")["PM2.5"]
        .apply(lambda x: x.isna().sum())
)

Missing PM2.5 dates by year:
Year
2021    17
2022    28
2023     6
2024    13
2025     3
Name: PM2.5, dtype: int64


Display dates with no usable measurement

In [17]:
missing_dates = data.loc[
    data["PM2.5"].isna(), "Date"
]

print("\nFirst 20 missing dates:")
print(missing_dates.head(20).to_list())


First 20 missing dates:
[Timestamp('2021-01-01 00:00:00'), Timestamp('2021-01-02 00:00:00'), Timestamp('2021-01-03 00:00:00'), Timestamp('2021-01-04 00:00:00'), Timestamp('2021-01-05 00:00:00'), Timestamp('2021-01-06 00:00:00'), Timestamp('2021-05-03 00:00:00'), Timestamp('2021-05-04 00:00:00'), Timestamp('2021-05-05 00:00:00'), Timestamp('2021-05-06 00:00:00'), Timestamp('2021-05-07 00:00:00'), Timestamp('2021-05-08 00:00:00'), Timestamp('2021-05-09 00:00:00'), Timestamp('2021-05-10 00:00:00'), Timestamp('2021-05-11 00:00:00'), Timestamp('2021-05-12 00:00:00'), Timestamp('2021-12-08 00:00:00'), Timestamp('2022-01-03 00:00:00'), Timestamp('2022-01-04 00:00:00'), Timestamp('2022-01-05 00:00:00')]


Inspect the methods recorded in the filtered data

In [18]:
print("\nMeasurement methods selected for this export:")
print(selected_methods.to_string(index=False))



Measurement methods selected for this export:
 POC  Method Code                                                                        Method Name
  23          736                      Teledyne T640 at 5.0 LPM (Corrected) - Broadband spectroscopy
   3          636 Teledyne T640 at 5.0 LPM w/Network Data Alignment enabled - Broadband spectroscopy


Remove the temporary Year column

In [19]:
data = data.drop(columns=["Year"])

Save the cleaned dataset

In [ ]:
# Compare against the preserved original POC-3-only export before saving.
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
original_filename = output_dir / "albemarle_pm25_poc3_without_interpolation.csv"
assert not data["Date"].duplicated().any(), "Duplicate calendar dates found."
assert len(data) == 1826, "Unexpected calendar length."
if original_filename.exists():
    original = pd.read_csv(original_filename, parse_dates=["Date"])
    assert data["Date"].equals(original["Date"]), "Calendar dates changed."
    assert data["PM2.5"].isna().equals(original["PM2.5"].isna()), "Missing dates changed."
    later = data["Date"] >= cutoff
    assert np.allclose(
        data.loc[later, "PM2.5"], original.loc[later, "PM2.5"], equal_nan=True
    ), "Pollution values from August 2023 onward unexpectedly changed."
    changed = ~np.isclose(data["PM2.5"], original["PM2.5"], equal_nan=True)
    print("Older pollution values changed:", int(changed.sum()))
    print("Verified: newer pollution values and missing dates are unchanged.")

# Missing observations remain blank; the original comparison file is not overwritten.
output_filename = output_dir / "albemarle_pm25_corrected_poc_without_interpolation.csv"
data.to_csv(output_filename, index=False)
print("CSV saved:", output_filename)
print("Final shape:", data.shape)
print("Missing PM2.5 values:", data["PM2.5"].isna().sum())
print("Final columns:", data.columns.tolist())


Check the saved file location (no Colab download needed)

In [ ]:
print("CSV is available locally at:", output_filename.resolve())
